In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np

# 저장한 파일 찾기
matches = list(
    Path("/content/drive/MyDrive").rglob("인허가_폐업정보결합.csv")
)

for path in matches:
    print(path)

if len(matches) != 1:
    raise ValueError("파일이 0개이거나 여러 개입니다. 출력된 경로를 확인하세요.")

# 원본 보관
df_base = pd.read_csv(
    matches[0],
    encoding="utf-8-sig",
    dtype="string"
)

df = df_base.copy()

# 날짜·숫자 자료형 복원
for col in ["인허가일자_dt", "폐업일자_dt"]:
    df[col] = pd.to_datetime(
        df[col],
        format="ISO8601",
        errors="raise"
    )

for col in ["lon", "lat", "프랜차이즈유무"]:
    df[col] = pd.to_numeric(df[col], errors="raise")

print(f"\n데이터: {df.shape[0]:,}행 × {df.shape[1]}개 컬럼")
print("점포ID 결측:", df["점포ID"].isna().sum())
print("점포ID 중복:", df["점포ID"].duplicated().sum())
print("점포이력ID 결측:", df["점포이력ID"].isna().sum())
print("고유 점포이력ID:", df["점포이력ID"].nunique())

display(df.head())

In [ ]:
df.head()

In [ ]:
# 집계용 데이터
status_check = df[
    ["점포이력ID", "점포ID", "영업여부", "폐업일자_dt"]
].copy()

status_check["영업여부"] = (
    status_check["영업여부"].astype("string").str.strip()
)

status_check["영업건"] = status_check["영업여부"].eq("T").fillna(False)
status_check["폐업건"] = status_check["영업여부"].eq("F").fillna(False)
status_check["상태확인필요건"] = ~status_check["영업여부"].isin(["T", "F"])

store_status = (
    status_check.groupby("점포이력ID")
    .agg(
        인허가건수=("점포ID", "size"),
        영업건수=("영업건", "sum"),
        폐업건수=("폐업건", "sum"),
        상태확인필요건수=("상태확인필요건", "sum"),
        최초폐업일=("폐업일자_dt", "min"),
        최종폐업일=("폐업일자_dt", "max")
    )
    .reset_index()
)

store_status["상태유형"] = np.select(
    [
        store_status["상태확인필요건수"].gt(0),
        store_status["영업건수"].eq(store_status["인허가건수"]),
        store_status["폐업건수"].eq(store_status["인허가건수"])
    ],
    [
        "상태 확인 필요",
        "전부 영업",
        "전부 폐업"
    ],
    default="영업·폐업 혼합"
)

display(
    store_status.groupby("상태유형")
    .agg(
        점포수=("점포이력ID", "size"),
        인허가행수=("인허가건수", "sum")
    )
    .reset_index()
)

display(store_status.head(20))

In [ ]:
multi_permit_status = store_status.loc[
    store_status["인허가건수"] >= 2
].copy()

display(
    multi_permit_status["상태유형"]
    .value_counts()
    .rename_axis("상태유형")
    .reset_index(name="점포수")
)

In [ ]:
mixed_ids = store_status.loc[
    store_status["상태유형"] == "영업·폐업 혼합",
    "점포이력ID"
]

mixed_rows = (
    df.loc[df["점포이력ID"].isin(mixed_ids)]
    .sort_values(["점포이력ID", "인허가일자_dt", "점포ID"])
    .copy()
)

print(f"영업·폐업 혼합 점포: {len(mixed_ids):,}개")
print(f"해당 인허가 행: {len(mixed_rows):,}건")

with pd.option_context("display.max_columns", None):
    display(mixed_rows.head(50))

In [ ]:
# 기존 df는 변경하지 않음
work = df.copy()
work["_상태"] = work["영업여부"].astype("string").str.strip()

# T와 F가 모두 있는 점포이력ID
status_sets = work.groupby("점포이력ID")["_상태"].agg(
    lambda x: set(x.dropna())
)

mixed_ids = status_sets[
    status_sets.apply(lambda x: {"T", "F"}.issubset(x))
].index

mixed_timeline = work.loc[
    work["점포이력ID"].isin(mixed_ids)
].copy()

# 시간 제외, 날짜 기준 비교
mixed_timeline["_등록일"] = (
    mixed_timeline["인허가일자_dt"].dt.normalize()
)
mixed_timeline["_폐업일"] = (
    mixed_timeline["폐업일자_dt"].dt.normalize()
)

# 점포별 등록 날짜 유형
summary = (
    mixed_timeline.groupby("점포이력ID")
    .agg(
        인허가건수=("점포ID", "size"),
        등록날짜수=("_등록일", "nunique"),
        등록일결측건수=("_등록일", lambda x: x.isna().sum())
    )
)

summary["등록유형"] = np.select(
    [
        summary["등록일결측건수"].gt(0),
        summary["등록날짜수"].eq(1)
    ],
    [
        "등록일 확인 필요",
        "동일 날짜 등록"
    ],
    default="여러 날짜에 순차 등록"
)

mixed_timeline = mixed_timeline.merge(
    summary[["등록유형"]],
    left_on="점포이력ID",
    right_index=True,
    how="left",
    validate="many_to_one"
)

display(
    summary["등록유형"]
    .value_counts()
    .rename_axis("등록유형")
    .reset_index(name="점포수")
)

In [ ]:
# 점포별 서로 다른 등록 날짜 목록
registration_dates = (
    mixed_timeline[["점포이력ID", "_등록일"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["점포이력ID", "_등록일"])
)

registration_dates["다음등록일"] = (
    registration_dates.groupby("점포이력ID")["_등록일"].shift(-1)
)

mixed_timeline = mixed_timeline.merge(
    registration_dates,
    on=["점포이력ID", "_등록일"],
    how="left",
    validate="many_to_one"
)

has_next = mixed_timeline["다음등록일"].notna()
is_closed = mixed_timeline["_상태"].eq("F").fillna(False)
has_close = mixed_timeline["_폐업일"].notna()

mixed_timeline["날짜비교"] = np.select(
    [
        mixed_timeline["_등록일"].isna(),
        ~has_next,
        is_closed & ~has_close,
        is_closed & has_close
        & (mixed_timeline["_폐업일"] < mixed_timeline["다음등록일"]),
        is_closed & has_close
        & (mixed_timeline["_폐업일"] == mixed_timeline["다음등록일"]),
        is_closed & has_close
        & (mixed_timeline["_폐업일"] > mixed_timeline["다음등록일"]),
        mixed_timeline["_상태"].eq("T").fillna(False)
    ],
    [
        "등록일 확인 필요",
        "후속 등록 없음",
        "폐업일 확인 필요",
        "폐업 후 후속 등록",
        "폐업일과 후속 등록일 같음",
        "후속 등록 당시 미폐업",
        "현재 영업 인허가에 후속 등록 존재"
    ],
    default="상태 확인 필요"
)

# 양수: 폐업 후 다음 등록까지 날짜 차이
# 음수: 다음 등록보다 뒤에 폐업
mixed_timeline["폐업후_다음등록까지_일수"] = (
    mixed_timeline["다음등록일"] - mixed_timeline["_폐업일"]
).dt.days

cols = [
    "점포이력ID", "점포ID", "사업장명", "업종명",
    "등록유형", "인허가일자_dt", "폐업일자_dt", "영업여부",
    "다음등록일", "폐업후_다음등록까지_일수", "날짜비교"
]

mixed_timeline = mixed_timeline.sort_values(
    ["점포이력ID", "_등록일", "점포ID"]
)

with pd.option_context("display.max_columns", None):
    display(mixed_timeline[cols].head(50))

In [ ]:
mixed_timeline["날짜비교"].value_counts(dropna=False)

In [ ]:
# 같은 점포이력ID가 2행 이상인 경우
mask = (
    df["점포이력ID"].notna()
    & df["점포이력ID"].duplicated(keep=False)
)

multi_permit_rows = (
    df.loc[
        mask,
        [
            "점포이력ID",
            "점포ID",
            "사업장명",
            "업종명",
            "업태구분명",
            "영업여부",
            "인허가일자_dt",
            "폐업일자_dt"
        ]
    ]
    .sort_values(["점포이력ID", "인허가일자_dt", "점포ID"])
    .copy()
)

print(f"점포 수: {multi_permit_rows['점포이력ID'].nunique():,}개")
print(f"인허가 행 수: {len(multi_permit_rows):,}건")

with pd.option_context(
    "display.max_columns", None,
    "display.max_rows", 100
):
    display(multi_permit_rows)


In [ ]:
summary = df.groupby("점포이력ID").agg(
    인허가건수=("점포ID", "nunique"),
    업종수=("업종명", "nunique"),
    전부영업=("영업여부", lambda x: x.eq("T").fillna(False).all()),
    업종목록=("업종명", lambda x: sorted(x.dropna().unique()))
)

display(
    summary.loc[
        summary["인허가건수"].gt(1) & summary["전부영업"]
    ]
)

In [ ]:
work = df.copy()

work["_상태"] = (
    work["영업여부"].astype("string").str.strip()
)
work["_등록일"] = work["인허가일자_dt"].dt.normalize()

# 1. 연결된 인허가가 전부 영업 중인 점포
all_open = work.groupby("점포이력ID")["_상태"].agg(
    lambda x: x.eq("T").fillna(False).all()
)

all_open_ids = all_open[all_open].index

open_rows = work.loc[
    work["점포이력ID"].isin(all_open_ids)
].copy()

# 2. 같은 점포·같은 날짜에 서로 다른 업종이 2개 이상인지 확인
same_day_summary = (
    open_rows.groupby(["점포이력ID", "_등록일"])["업종명"]
    .nunique()
    .reset_index(name="업종수")
)

same_day_multi = same_day_summary.loc[
    same_day_summary["업종수"] >= 2
].copy()

# 3. 조건에 해당하는 날짜의 인허가 행 조회
same_day_multi_rows = (
    open_rows.merge(
        same_day_multi,
        on=["점포이력ID", "_등록일"],
        how="inner",
        validate="many_to_one"
    )
    .sort_values(["점포이력ID", "_등록일", "업종명", "점포ID"])
)

print(
    f"해당 점포: "
    f"{same_day_multi_rows['점포이력ID'].nunique():,}개"
)
print(f"해당 인허가 행: {len(same_day_multi_rows):,}건")

cols = [
    "점포이력ID", "점포ID", "사업장명",
    "업종명", "카드업종", "업태구분명",
    "인허가일자_dt", "영업여부", "업종수"
]

with pd.option_context(
    "display.max_columns", None,
    "display.max_rows", 100
):
    display(same_day_multi_rows[cols])

In [ ]:
from pathlib import Path

save_dir = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6/인허가_분석결과"
)
save_dir.mkdir(parents=True, exist_ok=True)

stamp = pd.Timestamp.now(tz="Asia/Seoul").strftime("%Y%m%d_%H%M%S")
save_path = save_dir / f"인허가_작업본_{stamp}.csv"

df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"저장 완료: {df.shape[0]:,}행 × {df.shape[1]}개 컬럼")
print("저장 경로:", save_path)